Cell 1: setup

In [1]:
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.width", 200)

RANDOM_STATE = 42
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

FEATURES = ["wfo", "wwr", "sdt", "sdn", "sdd", "sda"]
CATEGORICAL = ["wfo", "sdt"]

Cell 2: load the cleaned data

In [2]:
data = pd.read_csv("clean_cooling.csv")
print(data.shape)
print(data.dtypes)
data.head()

(11760, 7)
wfo          int64
wwr        float64
sdt            str
sdn          int64
sdd        float64
sda          int64
cooling    float64
dtype: object


,wfo,wwr,sdt,sdn,sdd,sda,cooling
0,0,0.15,HSD,1,0.3,90,1.081554e+11
1,45,0.15,HSD,1,0.3,90,1.080874e+11
2,90,0.15,HSD,1,0.3,90,1.080167e+11
3,135,0.15,HSD,1,0.3,90,1.081876e+11
4,180,0.15,HSD,1,0.3,90,1.083706e+11


Cell 3: build the index target and split

In [3]:
BASE = data["cooling"].mean()          # saved to metadata later, the app needs it
data["cooling_index"] = data["cooling"] / BASE * 100

X = data[FEATURES]
y = data["cooling_index"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("BASE:", BASE)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("\nIndex target summary:")
print(y.describe().round(3))

# The split should look like the full data: compare shading-type shares
print("\nSDT share (full / train / test):")
print(pd.concat([
    X["sdt"].value_counts(normalize=True),
    X_train["sdt"].value_counts(normalize=True),
    X_test["sdt"].value_counts(normalize=True),
], axis=1, keys=["full", "train", "test"]).round(3))

BASE: 108805094365.37576
Train: (9408, 6)  Test: (2352, 6)

Index target summary:
count    11760.000
mean       100.000
std          0.482
min         99.031
25%         99.661
50%         99.883
75%        100.218
max        102.422
Name: cooling_index, dtype: float64

SDT share (full / train / test):
      full  train  test
sdt                    
VSD  0.643  0.644  0.64
HSD  0.357  0.356  0.36


Cell 4: model-building function

In [4]:
def build_pipeline(model):
    """Return a fresh pipeline (own encoder) for the given model."""
    prep = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)],
        remainder="passthrough",
    )
    return Pipeline([("prep", prep), ("model", model)])

def make_models():
    return {
        "Mean baseline": DummyRegressor(strategy="mean"),
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(
            n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1
        ),
    }

# Quick check that the pipeline wiring works (fits only the cheap baseline)
test_pipe = build_pipeline(DummyRegressor(strategy="mean")).fit(X_train, y_train)
print("Pipeline OK, example prediction:", test_pipe.predict(X_test.head(1)))

Pipeline OK, example prediction: [99.99844896]


Cell 5: train all three models and compare

In [5]:
fitted, preds, rows = {}, {}, []

for name, model in make_models().items():
    pipe = build_pipeline(model).fit(X_train, y_train)
    p = pipe.predict(X_test)
    fitted[name] = pipe
    preds[name] = p
    rows.append({
        "model": name,
        "MAE": mean_absolute_error(y_test, p),
        "RMSE": np.sqrt(mean_squared_error(y_test, p)),
        "R2": r2_score(y_test, p),
    })

results = pd.DataFrame(rows).set_index("model").round(4)
print(results)

                      MAE    RMSE      R2
model                                    
Mean baseline      0.3635  0.4879 -0.0003
Linear Regression  0.1973  0.2646  0.7058
Random Forest      0.0403  0.0612  0.9843


Cell 6: does the model overfit? (train vs test)

In [6]:
for name in ["Linear Regression", "Random Forest"]:
    tr = r2_score(y_train, fitted[name].predict(X_train))
    te = r2_score(y_test, fitted[name].predict(X_test))
    print(f"{name}: train R2 = {tr:.4f}, test R2 = {te:.4f}")

Linear Regression: train R2 = 0.7179, test R2 = 0.7058
Random Forest: train R2 = 0.9977, test R2 = 0.9843


Cell 7: pairwise accuracy by true gap

In [7]:
rng = np.random.default_rng(RANDOM_STATE)
yt = y_test.to_numpy()

i = rng.integers(0, len(yt), 100000)
j = rng.integers(0, len(yt), 100000)
keep = i != j
i, j = i[keep], j[keep]

true_d = yt[i] - yt[j]
bins = [0, 0.05, 0.1, 0.2, 0.3, 0.5, 1, np.inf]

for name in ["Linear Regression", "Random Forest"]:
    pred_d = preds[name][i] - preds[name][j]
    res = pd.DataFrame({
        "true_gap": pd.cut(np.abs(true_d), bins=bins, include_lowest=True),
        "correct": np.sign(true_d) == np.sign(pred_d),
    })
    print(name)
    print(res.groupby("true_gap", observed=True)["correct"]
             .agg(["mean", "count"]).round(3))
    print()

Linear Regression
                 mean  count
true_gap                    
(-0.001, 0.05]  0.540   7532
(0.05, 0.1]     0.605   7403
(0.1, 0.2]      0.708  14181
(0.2, 0.3]      0.819  12467
(0.3, 0.5]      0.918  19683
(0.5, 1.0]      0.975  25122
(1.0, inf]      0.998  13572

Random Forest
                 mean  count
true_gap                    
(-0.001, 0.05]  0.662   7532
(0.05, 0.1]     0.846   7403
(0.1, 0.2]      0.950  14181
(0.2, 0.3]      0.988  12467
(0.3, 0.5]      0.999  19683
(0.5, 1.0]      1.000  25122
(1.0, inf]      1.000  13572



Cell 8: hold out all WWR = 0.30 rows

In [8]:
mask = np.isclose(data["wwr"], 0.30)
X_tr, y_tr = X[~mask], y[~mask]
X_ho, y_ho = X[mask], y[mask]
print("Train rows:", len(X_tr), " Held-out (WWR=0.30) rows:", len(X_ho))

rows = []
for name, model in make_models().items():
    pipe = build_pipeline(model).fit(X_tr, y_tr)
    p = pipe.predict(X_ho)
    rows.append({
        "model": name,
        "MAE": mean_absolute_error(y_ho, p),
        "RMSE": np.sqrt(mean_squared_error(y_ho, p)),
        "R2": r2_score(y_ho, p),
    })

print(pd.DataFrame(rows).set_index("model").round(4))

Train rows: 10080  Held-out (WWR=0.30) rows: 1680
                      MAE    RMSE      R2
model                                    
Mean baseline      0.2751  0.3502 -0.0068
Linear Regression  0.1580  0.2023  0.6639
Random Forest      0.1713  0.1929  0.6944


Cell 9: leave-one-WWR-level-out, several models

In [9]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor

def poly_pipeline():
    prep = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL)],
        remainder="passthrough",
    )
    return Pipeline([
        ("prep", prep),
        ("poly", PolynomialFeatures(2, include_bias=False)),
        ("scale", StandardScaler()),
        ("model", Ridge(alpha=1.0)),
    ])

candidates = {
    "Mean baseline":      lambda: build_pipeline(DummyRegressor(strategy="mean")),
    "Linear Regression":  lambda: build_pipeline(LinearRegression()),
    "Ridge + interactions": poly_pipeline,
    "Random Forest":      lambda: build_pipeline(RandomForestRegressor(
                              n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
    "Gradient Boosting":  lambda: build_pipeline(GradientBoostingRegressor(
                              random_state=RANDOM_STATE)),
}

levels = [0.20, 0.25, 0.30, 0.35, 0.40]   # interior levels only (0.15/0.45 would be extrapolation)
table = {}
for name, make in candidates.items():
    row = {}
    for lv in levels:
        m = np.isclose(data["wwr"], lv)
        pipe = make().fit(X[~m], y[~m])
        row[lv] = np.sqrt(mean_squared_error(y[m], pipe.predict(X[m])))
    row["mean"] = np.mean(list(row.values()))
    table[name] = row

print("RMSE when each WWR level is held out (lower is better)")
print(pd.DataFrame(table).T.round(3))

RMSE when each WWR level is held out (lower is better)
                        0.2   0.25    0.3   0.35    0.4   mean
Mean baseline         0.407  0.336  0.350  0.449  0.607  0.430
Linear Regression     0.223  0.192  0.202  0.249  0.317  0.237
Ridge + interactions  0.124  0.112  0.124  0.148  0.179  0.137
Random Forest         0.153  0.165  0.193  0.195  0.194  0.180
Gradient Boosting     0.151  0.182  0.212  0.250  0.242  0.207


Cell 10: tune degree and alpha (leave-one-WWR-level-out on four levels)

In [10]:
def poly_pipeline(degree=2, alpha=1.0):
    prep = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL)],
        remainder="passthrough",
    )
    return Pipeline([
        ("prep", prep),
        ("poly", PolynomialFeatures(degree, include_bias=False)),
        ("scale", StandardScaler()),
        ("model", Ridge(alpha=alpha)),
    ])

tune_levels = [0.20, 0.25, 0.35, 0.40]   # 0.30 is kept aside as the final check
rows = []
for degree in [2, 3]:
    for alpha in [0.01, 0.1, 1, 10]:
        errs = []
        for lv in tune_levels:
            m = np.isclose(data["wwr"], lv)
            pipe = poly_pipeline(degree, alpha).fit(X[~m], y[~m])
            errs.append(np.sqrt(mean_squared_error(y[m], pipe.predict(X[m]))))
        rows.append({"degree": degree, "alpha": alpha, "mean_RMSE": np.mean(errs),
                     **{f"wwr_{lv}": e for lv, e in zip(tune_levels, errs)}})

tuning = pd.DataFrame(rows).sort_values("mean_RMSE")
print(tuning.round(3).to_string(index=False))

 degree  alpha  mean_RMSE  wwr_0.2  wwr_0.25  wwr_0.35  wwr_0.4
      3   0.10      0.105    0.098     0.088     0.109    0.126
      3   0.01      0.105    0.098     0.088     0.109    0.126
      3   1.00      0.105    0.097     0.087     0.110    0.127
      3  10.00      0.107    0.098     0.087     0.113    0.132
      2  10.00      0.141    0.123     0.111     0.148    0.180
      2   1.00      0.141    0.124     0.112     0.148    0.179
      2   0.10      0.141    0.124     0.112     0.148    0.179
      2   0.01      0.141    0.124     0.112     0.148    0.179


Cell 11: final check on WWR = 0.30, plus the random split

In [11]:
best = tuning.iloc[0]
best_degree, best_alpha = int(best["degree"]), float(best["alpha"])
print(f"Chosen: degree={best_degree}, alpha={best_alpha}\n")

# Untouched level
m = np.isclose(data["wwr"], 0.30)
pipe = poly_pipeline(best_degree, best_alpha).fit(X[~m], y[~m])
rmse_030 = np.sqrt(mean_squared_error(y[m], pipe.predict(X[m])))
print(f"WWR=0.30 held out -> RMSE {rmse_030:.3f}   (Random Forest was 0.193, Ridge deg2 was 0.124)")

# Standard random split
pipe = poly_pipeline(best_degree, best_alpha).fit(X_train, y_train)
p = pipe.predict(X_test)
print("\nRandom 80/20 split (Random Forest: MAE 0.0403, RMSE 0.0612, R2 0.9843):")
print(f"  MAE  {mean_absolute_error(y_test, p):.4f}")
print(f"  RMSE {np.sqrt(mean_squared_error(y_test, p)):.4f}")
print(f"  R2   {r2_score(y_test, p):.4f}")

Chosen: degree=3, alpha=0.1

WWR=0.30 held out -> RMSE 0.096   (Random Forest was 0.193, Ridge deg2 was 0.124)

Random 80/20 split (Random Forest: MAE 0.0403, RMSE 0.0612, R2 0.9843):
  MAE  0.0794
  RMSE 0.1104
  R2   0.9488


Cell 12: held-out predictions for both models (every WWR level held out in turn)

In [12]:
rf_maker = lambda: build_pipeline(RandomForestRegressor(
    n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1))

parts = []
for lv in [0.20, 0.25, 0.30, 0.35, 0.40]:
    m = np.isclose(data["wwr"], lv)
    part = X[m].copy()
    part["y"] = y[m].to_numpy()
    part["level"] = lv
    part["Ridge deg3"] = poly_pipeline(3, 0.1).fit(X[~m], y[~m]).predict(X[m])
    part["Random Forest"] = rf_maker().fit(X[~m], y[~m]).predict(X[m])
    parts.append(part)

ho = pd.concat(parts, ignore_index=True)

# sanity check: Ridge should be ~0.088-0.126 per level, RF ~0.15-0.20
for name in ["Ridge deg3", "Random Forest"]:
    print(name, ho.groupby("level").apply(
        lambda g: np.sqrt(mean_squared_error(g["y"], g[name]))).round(3).to_dict())

Ridge deg3 {0.2: 0.098, 0.25: 0.088, 0.3: 0.096, 0.35: 0.109, 0.4: 0.126}
Random Forest {0.2: 0.153, 0.25: 0.165, 0.3: 0.193, 0.35: 0.195, 0.4: 0.194}


Cell 13: pairwise accuracy, random pairs and one-input-change pairs

In [13]:
from itertools import combinations

MODELS = ["Ridge deg3", "Random Forest"]
bins = [0, 0.05, 0.1, 0.2, 0.3, 0.5, np.inf]
yv = ho["y"].to_numpy()

def report(i, j, title):
    print(f"=== {title}  ({len(i):,} pairs) ===")
    true_d = yv[i] - yv[j]
    for name in MODELS:
        pv = ho[name].to_numpy()
        pred_d = pv[i] - pv[j]
        correct = np.sign(true_d) == np.sign(pred_d)
        print(f"\n{name}: overall accuracy {correct.mean():.3f}")
        by_pred = pd.DataFrame({"pred_gap": pd.cut(np.abs(pred_d), bins=bins, include_lowest=True),
                                "correct": correct})
        print("  grouped by PREDICTED gap (what the app can see):")
        print(by_pred.groupby("pred_gap", observed=True)["correct"].agg(["mean", "count"]).round(3).to_string())
    print()

# A) random pairs inside the same held-out WWR level
rng = np.random.default_rng(RANDOM_STATE)
ai, aj = [], []
for lv, g in ho.groupby("level"):
    idx = g.index.to_numpy()
    a, b = rng.choice(idx, 20000), rng.choice(idx, 20000)
    k = a != b
    ai.append(a[k]); aj.append(b[k])
report(np.concatenate(ai), np.concatenate(aj), "Random pairs, same held-out WWR")

# B) pairs that differ in exactly ONE of orientation / quantity / depth / tilt
bi, bj = [], []
for c in ["wfo", "sdn", "sdd", "sda"]:
    others = [f for f in FEATURES if f != c]
    for _, g in ho.groupby(others):
        for a, b in combinations(g.index.to_numpy(), 2):
            bi.append(a); bj.append(b)
report(np.array(bi), np.array(bj), "One-input-change pairs (A vs revised B)")

=== Random pairs, same held-out WWR  (99,934 pairs) ===

Ridge deg3: overall accuracy 0.894
  grouped by PREDICTED gap (what the app can see):
                 mean  count
pred_gap                    
(-0.001, 0.05]  0.584  10727
(0.05, 0.1]     0.720  10067
(0.1, 0.2]      0.864  18025
(0.2, 0.3]      0.953  15294
(0.3, 0.5]      0.992  21145
(0.5, inf]      1.000  24676

Random Forest: overall accuracy 0.920
  grouped by PREDICTED gap (what the app can see):
                 mean  count
pred_gap                    
(-0.001, 0.05]  0.653  12936
(0.05, 0.1]     0.833  11348
(0.1, 0.2]      0.938  18890
(0.2, 0.3]      0.976  13929
(0.3, 0.5]      0.996  18283
(0.5, inf]      1.000  24548

=== One-input-change pairs (A vs revised B)  (82,200 pairs) ===

Ridge deg3: overall accuracy 0.795
  grouped by PREDICTED gap (what the app can see):
                 mean  count
pred_gap                    
(-0.001, 0.05]  0.616  28406
(0.05, 0.1]     0.773  15977
(0.1, 0.2]      0.886  16764
(0.2, 

Cell 14: WWR-change pairs and finer bins

In [14]:
others = ["wfo", "sdt", "sdn", "sdd", "sda"]
names = ["Ridge deg3", "Random Forest", "Average of both"]
fine_bins = [0, 0.025, 0.05, 0.075, 0.1, 0.15, 0.2, 0.3, np.inf]

frames = []
for lv in [0.20, 0.25, 0.30, 0.35, 0.40]:
    m = np.isclose(data["wwr"], lv)
    ridge = poly_pipeline(3, 0.1).fit(X[~m], y[~m])
    forest = rf_maker().fit(X[~m], y[~m])

    base = data[others + ["wwr"]].copy()
    base["y"] = y
    base["Ridge deg3"] = ridge.predict(X)
    base["Random Forest"] = forest.predict(X)
    base["Average of both"] = (base["Ridge deg3"] + base["Random Forest"]) / 2

    held = base[m]                      # B: held-out WWR level (acts as "off-grid")
    for nb in [round(lv - 0.05, 2), round(lv + 0.05, 2)]:
        neighbour = base[np.isclose(base["wwr"], nb)]   # A: on-grid, seen in training
        frames.append(held.merge(neighbour, on=others, suffixes=("_B", "_A")))

pairs = pd.concat(frames, ignore_index=True)
true_d = pairs["y_B"] - pairs["y_A"]
print(f"{len(pairs):,} pairs (A on-grid WWR, B at unseen WWR, all else identical)")
print("Typical true gap:", round(true_d.abs().median(), 3), "index points\n")

for name in names:
    pred_d = pairs[f"{name}_B"] - pairs[f"{name}_A"]
    correct = np.sign(true_d) == np.sign(pred_d)
    print(f"{name}: overall accuracy {correct.mean():.3f}")
    out = (pd.DataFrame({"pred_gap": pd.cut(pred_d.abs(), bins=fine_bins, include_lowest=True),
                         "correct": correct})
           .groupby("pred_gap", observed=True)["correct"].agg(["mean", "count"]).round(3))
    print(out.to_string(), "\n")

16,800 pairs (A on-grid WWR, B at unseen WWR, all else identical)
Typical true gap: 0.146 index points

Ridge deg3: overall accuracy 0.936
                  mean  count
pred_gap                     
(-0.001, 0.025]  0.811    438
(0.025, 0.05]    0.832   1042
(0.05, 0.075]    0.791   1500
(0.075, 0.1]     0.862   1911
(0.1, 0.15]      0.959   4758
(0.15, 0.2]      0.988   3277
(0.2, 0.3]       0.999   2919
(0.3, inf]       1.000    955 

Random Forest: overall accuracy 0.473
                  mean  count
pred_gap                     
(-0.001, 0.025]  0.013   8611
(0.025, 0.05]    0.590    205
(0.05, 0.075]    0.678    230
(0.075, 0.1]     0.784    241
(0.1, 0.15]      0.920    758
(0.15, 0.2]      0.959   1122
(0.2, 0.3]       0.984   1912
(0.3, inf]       0.998   3721 

Average of both: overall accuracy 0.939
                  mean  count
pred_gap                     
(-0.001, 0.025]  0.736   1044
(0.025, 0.05]    0.820   1915
(0.05, 0.075]    0.929   2695
(0.075, 0.1]     0.962   2264

Cell 15: set the threshold from the held-out forest results

In [15]:
bi_arr, bj_arr = np.array(bi), np.array(bj)
pv = ho["Random Forest"].to_numpy()

pred_gap = np.abs(pv[bi_arr] - pv[bj_arr])
correct = np.sign(yv[bi_arr] - yv[bj_arr]) == np.sign(pv[bi_arr] - pv[bj_arr])

rows = []
for t in [0.0, 0.025, 0.05, 0.075, 0.1, 0.15, 0.2, 0.3]:
    sel = pred_gap >= t
    rows.append({"predicted gap >=": t,
                 "share of pairs": sel.mean(),
                 "accuracy": correct[sel].mean()})
table = pd.DataFrame(rows).round(3)
print(table.to_string(index=False))

ok = table[table["accuracy"] >= 0.90]
THRESHOLD = float(ok["predicted gap >="].iloc[0]) if len(ok) else None
print("\nSmallest predicted gap where accuracy is >= 90%:", THRESHOLD)

 predicted gap >=  share of pairs  accuracy
            0.000           1.000     0.885
            0.025           0.768     0.938
            0.050           0.634     0.955
            0.075           0.537     0.965
            0.100           0.460     0.971
            0.150           0.344     0.979
            0.200           0.264     0.988
            0.300           0.170     0.996

Smallest predicted gap where accuracy is >= 90%: 0.025


Cell 16: train the final model on all data and save it

In [16]:
import sklearn, datetime

final_model = build_pipeline(RandomForestRegressor(
    n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)).fit(X, y)

hsd = data[data["sdt"] == "HSD"]
vsd = data[data["sdt"] == "VSD"]
def space(d):
    return {"sdn": sorted(d["sdn"].unique().tolist()),
            "sdd": sorted(d["sdd"].unique().tolist()),
            "sda": sorted(d["sda"].unique().tolist())}

meta = {
    "model": "RandomForestRegressor (300 trees) in a one-hot pipeline",
    "target": "cooling index (dataset mean cooling = 100)",
    "BASE": float(BASE),
    "features": FEATURES,
    "wfo_values": sorted(data["wfo"].unique().tolist()),
    "wwr_values": sorted(data["wwr"].round(2).unique().tolist()),
    "design_space": {"HSD": space(hsd), "VSD": space(vsd)},
    "inconclusive_threshold": THRESHOLD,
    "clear_difference_threshold": 0.2,
    "heldout_random_split": {"MAE": 0.0403, "RMSE": 0.0612, "R2": 0.9843},
    "note": "Index relative to dataset mean; units not verified. Shanghai weather, BESTEST 600FF, grid values only.",
    "trained_on": str(datetime.date.today()),
    "sklearn_version": sklearn.__version__,
}

joblib.dump(final_model, ARTIFACT_DIR / "glazewise_model.joblib")
with open(ARTIFACT_DIR / "glazewise_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print("Saved:", [p.name for p in ARTIFACT_DIR.iterdir()])

Saved: ['glazewise_meta.json', 'glazewise_model.joblib']


Cell 17: reload check (what the app will do)

In [17]:
model2 = joblib.load(ARTIFACT_DIR / "glazewise_model.joblib")
meta2 = json.load(open(ARTIFACT_DIR / "glazewise_meta.json"))

sample = pd.DataFrame([{"wfo": 180, "wwr": 0.30, "sdt": "HSD", "sdn": 2, "sdd": 0.6, "sda": 90}])[meta2["features"]]
print("Same predictions after reload:", np.allclose(final_model.predict(X.head(50)), model2.predict(X.head(50))))
print("Example prediction (index):", model2.predict(sample)[0])
print("Allowed HSD options:", meta2["design_space"]["HSD"])

Same predictions after reload: True
Example prediction (index): 99.81267078548517
Allowed HSD options: {'sdn': [1, 2, 4], 'sdd': [0.3, 0.6, 0.9, 1.2, 1.5], 'sda': [90, 105, 120, 135, 150]}


Cell 18: update the saved metadata (no retraining)

In [19]:
meta2 = json.load(open(ARTIFACT_DIR / "glazewise_meta.json"))

FLOOR_AREA_M2 = 48.0
J_PER_KWH = 3.6e6
kwh_m2_per_index_point = meta2["BASE"] / J_PER_KWH / FLOOR_AREA_M2 / 100

wfo_labels = {"0": "East", "45": "South-East", "90": "South", "135": "South-West",
              "180": "West", "225": "North-West", "270": "North", "315": "North-East"}

# Model input = file value. Dropdown label = display value.
sda_display = {
    "HSD": {str(v): v - 60 for v in meta2["design_space"]["HSD"]["sda"]},
    "VSD": {str(v): v for v in meta2["design_space"]["VSD"]["sda"]},
}

meta2.update({
    "inconclusive_threshold": 0.05,
    "clear_difference_threshold": 0.2,
    "threshold_basis": "held-out one-input-change pairs, RF accuracy >=95.5% at predicted gap >= 0.05",
    "floor_area_m2": FLOOR_AREA_M2,
    "kwh_m2_per_index_point": kwh_m2_per_index_point,
    "unit_note": ("Raw values are joules for the whole room; converted as J/3.6e6/48. "
                  "Heating converts to ~696 kWh/m2 and matches the paper. Cooling converts to ~630 "
                  "but the paper reports ~266, so cooling is used as a relative index. "
                  "Index spread (std 0.48%) matches the paper's CED std (~0.5%)."),
    "period": "annual (paper, Fig. 12)",
    "wfo_labels": wfo_labels,
    "wfo_labels_verified": "Yi et al. 2024, Table 3",
    "sda_display": sda_display,
    "sda_note": "HSD file values are real angle + 60 (Table 3). VSD file values are used as is. Angle is from the window plane.",
    "baseline_note": "No unshaded case in the data. The paper's baseline (unshaded, west, 40% WWR) is not in combined_ed.",
})

with open(ARTIFACT_DIR / "glazewise_meta.json", "w") as f:
    json.dump(meta2, f, indent=2)

print(f"1 index point = {kwh_m2_per_index_point:.2f} kWh/m2 = {kwh_m2_per_index_point*FLOOR_AREA_M2:.0f} kWh/yr (room)")
print("HSD tilt labels:", sda_display["HSD"])

1 index point = 6.30 kWh/m2 = 302 kWh/yr (room)
HSD tilt labels: {'90': 30, '105': 45, '120': 60, '135': 75, '150': 90}
